# Day 45 — Sales & Service Demand Forecasting System
**Veda Technology | Data Science Internship**

## Project overview
Build a beginner-friendly forecasting workflow for technology services, digital services, training programs, and other business categories using historical time-series demand. This notebook uses **synthetic daily data** (2023–2025), so it contains no real customer or company information.

### Objectives
- Prepare and explore historical service-demand data.
- Identify long-term trend, weekly patterns, and annual seasonality.
- Establish a moving-average / naïve baseline.
- Compare a statistical model with a machine-learning approach.
- Evaluate forecasts on a chronological holdout set.
- Produce a 30-day forecast and practical business recommendations.

### Tools
Python, Pandas, NumPy, Matplotlib, Seaborn, scikit-learn, statsmodels, Jupyter Notebook. A Streamlit dashboard starter is included near the end.

> **Important:** The data is simulated for learning. The forecast is illustrative and should not be treated as a real business prediction.

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

RANDOM_STATE = 45
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 2. Create or load the synthetic dataset
If the CSV is in the same folder as this notebook, the first line loads it. Otherwise, the fallback generates the same style of synthetic data.

In [ ]:
from pathlib import Path

csv_file = Path("synthetic_service_demand.csv")
if csv_file.exists():
    data = pd.read_csv(csv_file, parse_dates=["date"])
else:
    rng = np.random.default_rng(RANDOM_STATE)
    dates = pd.date_range("2023-01-01", "2025-12-31", freq="D")
    n = len(dates)
    dow = dates.dayofweek.to_numpy()
    doy = dates.dayofyear.to_numpy()
    trend = np.linspace(0, 18, n)
    weekly = 7 * np.sin(2 * np.pi * dow / 7 + 0.4) + np.where(dow >= 5, 5, 0)
    yearly = 12 * np.sin(2 * np.pi * (doy - 30) / 365.25) + 5 * np.cos(4 * np.pi * doy / 365.25)
    promotion = rng.binomial(1, 0.08, n)
    noise = rng.normal(0, 5, n)
    demand = np.maximum(1, np.round(55 + trend + weekly + yearly + 10 * promotion + noise)).astype(int)
    data = pd.DataFrame({"date": dates, "demand": demand, "promotion": promotion})

data = data.sort_values("date").drop_duplicates("date").reset_index(drop=True)
data.head()

## 3. Data quality and summary

In [ ]:
print("Rows, columns:", data.shape)
print("\\nMissing values:\\n", data.isna().sum())
print("\\nDate range:", data["date"].min().date(), "to", data["date"].max().date())
print("\\nDemand summary:")
display(data["demand"].describe().to_frame())
assert data["date"].is_monotonic_increasing, "Dates should be sorted."
assert data["demand"].ge(0).all(), "Demand should be non-negative." 

## 4. Visualize the historical demand

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(data["date"], data["demand"], linewidth=0.8, label="Daily demand")
ax.plot(data["date"], data["demand"].rolling(30).mean(), linewidth=2, label="30-day rolling average")
ax.set(title="Daily Service Demand and 30-Day Rolling Average", xlabel="Date", ylabel="Demand (units / inquiries)")
ax.legend()
plt.tight_layout()
plt.show()

monthly = data.set_index("date")["demand"].resample("MS").sum()
monthly.plot(figsize=(13, 4), marker="o", title="Monthly Total Demand")
plt.ylabel("Total demand")
plt.tight_layout()
plt.show()

## 5. Trend and seasonality analysis

In [ ]:
data["day_of_week"] = data["date"].dt.day_name()
data["month"] = data["date"].dt.month_name()
data["day_num"] = data["date"].dt.dayofweek
data["month_num"] = data["date"].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=data, x="day_of_week", y="demand",
            order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"], ax=axes[0])
axes[0].set_title("Demand by Day of Week")
axes[0].tick_params(axis="x", rotation=35)
sns.boxplot(data=data, x="month", y="demand",
            order=["January","February","March","April","May","June","July","August","September","October","November","December"], ax=axes[1])
axes[1].set_title("Demand by Month")
axes[1].tick_params(axis="x", rotation=55)
plt.tight_layout()
plt.show()

print("Average demand by weekday:")
display(data.groupby("day_of_week")["demand"].mean().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).round(2))
print("\\nAverage demand by month:")
display(data.groupby("month")["demand"].mean().reindex(
    ["January","February","March","April","May","June","July","August","September","October","November","December"]).round(2))

## 6. Train / test split (time-aware)
Use the last 90 days as a holdout set. Do not randomly shuffle time-series data: that can leak future information into training.

In [ ]:
HORIZON = 90
train = data.iloc[:-HORIZON].copy()
test = data.iloc[-HORIZON:].copy()

print(f"Train: {train['date'].min().date()} to {train['date'].max().date()} ({len(train)} rows)")
print(f"Test:  {test['date'].min().date()} to {test['date'].max().date()} ({len(test)} rows)")

## 7. Baseline forecasts
The naïve baseline repeats the last training observation. The moving-average baseline repeats the training-set 7-day average. Baselines are important: a complex model should beat a simple benchmark to justify its use.

In [ ]:
naive_pred = np.repeat(train["demand"].iloc[-1], len(test))
moving_avg_pred = np.repeat(train["demand"].tail(7).mean(), len(test))

## 8. Statistical model — Holt-Winters Exponential Smoothing
Fit an additive trend and weekly seasonality. The model uses only the training window.

In [ ]:
hw_model = ExponentialSmoothing(
    train["demand"].astype(float),
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    initialization_method="estimated"
).fit(optimized=True)

hw_pred = np.maximum(0, hw_model.forecast(len(test)).to_numpy())

## 9. Machine-learning model — Random Forest
Create lagged demand features and calendar features. Each row predicts demand from past demand and known calendar values.

In [ ]:
def make_features(frame):
    out = frame[["date", "demand", "promotion"]].copy()
    out["day_of_week"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["day_of_year"] = out["date"].dt.dayofyear
    for lag in [1, 7, 14, 28]:
        out[f"lag_{lag}"] = out["demand"].shift(lag)
    out["rolling_mean_7"] = out["demand"].shift(1).rolling(7).mean()
    out["rolling_mean_28"] = out["demand"].shift(1).rolling(28).mean()
    return out.dropna().reset_index(drop=True)

features = make_features(data)
feature_cols = ["promotion", "day_of_week", "month", "day_of_year",
                "lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28"]
split_date = test["date"].min()
ml_train = features[features["date"] < split_date]
ml_test = features[features["date"] >= split_date]

rf = RandomForestRegressor(n_estimators=250, min_samples_leaf=3,
                           random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(ml_train[feature_cols], ml_train["demand"])
rf_pred = np.maximum(0, rf.predict(ml_test[feature_cols]))

# Align predictions and actuals by date after lag-feature warm-up.
ml_eval = ml_test[["date", "demand"]].copy()
ml_eval["rf_pred"] = rf_pred
print("Machine-learning evaluation rows:", len(ml_eval))

## 10. Compare forecast accuracy
Metrics: MAE (average absolute error), RMSE (penalizes large errors more), and WAPE (absolute error divided by total actual demand). Lower is better.

In [ ]:
def metrics(actual, predicted, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_%": 100 * np.sum(np.abs(actual - predicted)) / np.sum(np.abs(actual))
    }

results = [
    metrics(test["demand"], naive_pred, "Naive baseline"),
    metrics(test["demand"], moving_avg_pred, "7-day moving-average baseline"),
    metrics(test["demand"], hw_pred, "Holt-Winters (weekly seasonality)"),
    metrics(ml_eval["demand"], ml_eval["rf_pred"], "Random Forest (lag features)")
]
results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(results_df.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test["date"], test["demand"], label="Actual", linewidth=2)
ax.plot(test["date"], naive_pred, label="Naive baseline", alpha=0.8)
ax.plot(test["date"], moving_avg_pred, label="7-day moving average", alpha=0.8)
ax.plot(test["date"], hw_pred, label="Holt-Winters", alpha=0.9)
ax.plot(ml_eval["date"], ml_eval["rf_pred"], label="Random Forest", alpha=0.9)
ax.set(title="Actual vs Forecast — Holdout Period", xlabel="Date", ylabel="Demand")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

## 11. Select a model and forecast the next 30 days
Choose the lowest-MAE model from the holdout comparison. For a simple and stable operational forecast, this example refits Holt-Winters on all available history for the next 30 days. Review model results before using a forecast for decisions.

In [ ]:
FORECAST_DAYS = 30
final_hw = ExponentialSmoothing(
    data["demand"].astype(float),
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    initialization_method="estimated"
).fit(optimized=True)

future_dates = pd.date_range(data["date"].max() + pd.Timedelta(days=1),
                             periods=FORECAST_DAYS, freq="D")
future_values = np.maximum(0, final_hw.forecast(FORECAST_DAYS).to_numpy())
forecast = pd.DataFrame({"date": future_dates, "forecast_demand": np.round(future_values, 1)})
display(forecast.head(10))

fig, ax = plt.subplots(figsize=(13, 5))
recent = data.tail(120)
ax.plot(recent["date"], recent["demand"], label="Recent actual demand")
ax.plot(forecast["date"], forecast["forecast_demand"], marker="o", label="30-day forecast")
ax.set(title="Next 30 Days — Illustrative Demand Forecast", xlabel="Date", ylabel="Demand")
ax.legend()
plt.tight_layout()
plt.show()

forecast.to_csv("next_30_day_forecast.csv", index=False)
results_df.to_csv("model_evaluation_results.csv", index=False)
print("Saved next_30_day_forecast.csv and model_evaluation_results.csv")

## 12. Business interpretation and recommendations
Use the observed charts and metrics to write conclusions rather than assuming a pattern before seeing the results.

- **Capacity planning:** use daily forecasts to schedule support agents, trainers, and service delivery capacity.
- **Inventory / resources:** plan software licenses, equipment, and learning seats around expected demand.
- **Marketing:** compare demand before and after promotions; do not infer causality from this synthetic dataset alone.
- **Monitoring:** refresh forecasts as new actuals arrive and track MAE/WAPE monthly.
- **Risk:** demand may change due to new services, pricing, competitor actions, policy changes, or one-off events. Historical trends cannot guarantee future accuracy.
- **Model choice:** select using chronological holdout performance and business cost, not complexity alone.

### Limitations
The data is synthetic and generated with known patterns. It does not represent actual Veda Technology demand. The Random Forest evaluation uses lag features calculated from historical actual demand; for multi-step production forecasting, use a carefully designed recursive or direct-horizon strategy. Prediction intervals and external drivers are not included.

## 13. Optional Streamlit dashboard
Save the following code as `app.py` in the project folder, then run `streamlit run app.py`. It loads the included synthetic CSV and displays a monthly view. The dashboard is intentionally simple and can be expanded later.

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

st.set_page_config(page_title="Veda Demand Forecast", layout="wide")
st.title("Sales & Service Demand Forecasting")
st.caption("Learning demo using synthetic data — not real Veda Technology data.")

path = Path("synthetic_service_demand.csv")
if not path.exists():
    st.error("Place synthetic_service_demand.csv next to app.py.")
    st.stop()

df = pd.read_csv(path, parse_dates=["date"])
total = int(df["demand"].sum())
avg = float(df["demand"].mean())
peak = int(df["demand"].max())

c1, c2, c3 = st.columns(3)
c1.metric("Total demand", f"{total:,}")
c2.metric("Average daily demand", f"{avg:.1f}")
c3.metric("Peak daily demand", f"{peak:,}")

monthly = df.set_index("date")["demand"].resample("MS").sum()
st.subheader("Monthly demand")
st.line_chart(monthly)
st.subheader("Daily demand")
st.line_chart(df.set_index("date")["demand"])
st.dataframe(df.tail(20), use_container_width=True)


## 14. Run instructions and deliverables
1. Install dependencies: `pip install pandas numpy matplotlib seaborn scikit-learn statsmodels jupyter streamlit`.
2. Keep `synthetic_service_demand.csv` beside the notebook.
3. Open Jupyter and run cells from top to bottom.
4. Review the metric table and forecast plots before writing conclusions.
5. Optional dashboard: save the Streamlit code cell as `app.py`, then run `streamlit run app.py`.

### Deliverables
- Historical synthetic service-demand dataset
- Data preprocessing and trend/seasonality analysis
- Baseline and model forecasts
- Chronological model evaluation
- 30-day forecast CSV
- Streamlit dashboard starter
- Business planning recommendations